In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch


dataset = load_dataset(
    "dim/hendrycks_math_train_1k_DeepSeek-R1-Distill-Qwen-1.5B_max_len_4096_greedy"
)
dataset = dataset["train"].train_test_split(
    # test_size=250,
    test_size=350,
    # test_size=999,
    # test_size=1,
    seed=42,
)
dataset = dataset["test"].filter(lambda x: x["model_answer"].count("</think>") == 1)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    # attn_implementation="sdpa",
    attn_implementation="flash_attention_2",
)
model.requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Кодируем части текста в вектора

In [2]:
!pip install more_itertools -q

In [ ]:
[1, 2, 3, 4, 5, 6][-2:]

[5, 6]

In [7]:
from more_itertools import chunked
from tqdm import tqdm
import itertools

start_item = 100
cramming_tokens = []

for i in range(start_item, len(dataset)):
    print(f"{i}/{len(dataset)}")
    model_answer = dataset[i]["model_answer"]
    tokens = tokenizer.encode(
        dataset[i]["model_answer"],
        add_special_tokens=False,
    )
    mem_tokens = 8
    encode_size = mem_tokens * 4
    tokens_chunks = list(chunked(tokens, encode_size))
    # for chunk_part in range(2, len(tokens_chunks)):
    max_parts = 10
    for chunk_part in range(2, max_parts):
        train_part = tokens_chunks[:chunk_part]
        train_part = list(itertools.chain(*train_part))
        train_part_full_str = tokenizer.decode(
            train_part,
            add_special_tokens=False,
        )
        train_part_target_str = tokenizer.decode(
            train_part[-encode_size:][1:],
            add_special_tokens=False,
        )
        train_part = torch.tensor(
            train_part,
            device="cuda",
        ).unsqueeze(0)
        labels_part = train_part.clone()
        labels_part[:, :-encode_size] = -100
        # labels_part[]
        # print(train_part)
        # print(labels_part)
        compression_tensor_param = torch.nn.Parameter(
            torch.rand(
                mem_tokens,
                model.get_input_embeddings().weight.shape[1],
                device="cuda",
            ).unsqueeze(0),
            requires_grad=True,
        )
        optimizer = torch.optim.AdamW(
            [
                compression_tensor_param,
            ],
            lr=0.1,
        )
        compression_tensor = compression_tensor_param.repeat(
            1, encode_size // mem_tokens, 1
        )
        prev_tokens = train_part[:, :-encode_size].clone()
        prev_embeds = model.get_input_embeddings()(prev_tokens)
        input_embeds = torch.cat(
            [
                prev_embeds,
                compression_tensor,
            ],
            dim=1,
        )
        epoch_amount = 50
        dtype = torch.bfloat16
        for epoch in tqdm(range(epoch_amount)):
            compression_tensor = compression_tensor_param.repeat(
                1, encode_size // mem_tokens, 1
            )
            # print(compression_tensor_param)
            input_embeds = torch.cat(
                [
                    prev_embeds,
                    compression_tensor,
                ],
                dim=1,
            ).to(dtype)
            model_predicts = model(
                inputs_embeds=input_embeds,
                labels=labels_part,
            )

            compression_loss = model_predicts.loss
            compression_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            # print(
            #     "compression_loss",
            #     compression_loss,
            #     epoch,
            #     # len(tokenizer.encode(model_answer)),
            # )
        prediction_str = tokenizer.decode(
            model_predicts.logits.argmax(-1)[:, -encode_size:][-1]
        )
        # print("train_part_full_str", train_part_full_str)
        # print("train_part_target_str", train_part_target_str)
        # print("prediction_str", prediction_str)
        correct_reconstruction = (
            tokenizer.decode(
                model_predicts.logits.argmax(-1)[:, -encode_size:][-1][:-1]
            )
            == train_part_target_str
        )
        print(
            f"full text: '{tokenizer.decode(train_part[-1])}'",
        )
        print(
            f"context: '{tokenizer.decode(train_part[:, :-encode_size][-1])}'",
        )
        print(
            f"train_part_target_str: '{train_part_target_str}'",
        )
        print(f"correct_reconstruction: '{correct_reconstruction}'")
        print("=" * 50)
        print("=" * 50)
        cramming_tokens.append(compression_tensor_param.detach())
        # break

    break

100/209


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 35.78it/s]


full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
train_part_target_str: ' that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
correct_reconstruction: 'True'


100%|██████████| 50/50 [00:01<00:00, 35.70it/s]


...

full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a mul

100%|██████████| 50/50 [00:01<00:00, 34.81it/s]


full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the'
context: 'Okay, so I have this problem

100%|██████████| 50/50 [00:01<00:00, 32.98it/s]

full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the problem is trying to trick me into thin

In [ ]:
# 100/209
#   0%|          | 0/50 [00:00<?, ?it/s]100%|██████████| 50/50 [00:01<00:00, 35.78it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
# train_part_target_str: ' that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 35.70it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
# train_part_target_str: ' need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 35.78it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 3'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a'
# train_part_target_str: ' of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 3'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 35.55it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 3'
# train_part_target_str: ' the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 36.42it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that'
# train_part_target_str: ' be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 35.68it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1'
# train_part_target_str: '32, 32×2=64, 32×3=96, and so on. So, the first one is'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 34.81it/s]
# full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

# But wait, is there a trick here? Maybe the'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
# train_part_target_str: '32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

# But wait, is there a trick here? Maybe the'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================
# 100%|██████████| 50/50 [00:01<00:00, 32.98it/s]full text: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

# But wait, is there a trick here? Maybe the problem is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or'
# context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer.

# Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

# Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

# But wait, is there a trick here? Maybe the'
# train_part_target_str: ' is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or'
# correct_reconstruction: 'True'
# ==================================================
# ==================================================

In [170]:
tokenizer.decode(
    train_part[:, -encode_size:][-1][1:],
    # train_part[:, -encode_size:][-1][:],
    add_special_tokens=False,
)

' is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or'

In [175]:
tokenizer.decode(model_predicts.logits.argmax(-1)[:, -encode_size:][-1][:-1])

' is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or'

In [177]:
tokenizer.decode(model_predicts.logits.argmax(-1)[:, -encode_size:][-1])

' is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or different'

In [172]:
train_part_target_str

' is trying to trick me into thinking about something else. Sometimes, problems might ask for the smallest positive multiple in a different context, like in modular arithmetic or'

### Fixed train part

In [39]:
from more_itertools import chunked
from tqdm import tqdm
import itertools
import torch
import math

start_item = 100
cramming_tokens = []

for i in range(start_item, len(dataset)):
    print(f"{i}/{len(dataset)}")
    model_answer = dataset[i]["model_answer"]
    tokens = tokenizer.encode(
        dataset[i]["model_answer"],
        add_special_tokens=False,
    )
    mem_tokens = 8
    encode_size = mem_tokens * 4
    tokens_chunks = list(chunked(tokens, encode_size))
    max_parts = 10
    for chunk_part in range(2, max_parts):
        train_part = tokens_chunks[:chunk_part]
        train_part = list(itertools.chain(*train_part))

        train_part = torch.tensor(
            train_part,
            device="cuda",
        ).unsqueeze(0)

        target_len = encode_size + 1
        num_repeats = (target_len + mem_tokens - 1) // mem_tokens
        new_compression_len = num_repeats * mem_tokens

        context_len = train_part.shape[1] - encode_size
        if context_len < 0:
            continue

        labels_part = torch.full(
            (1, context_len + new_compression_len),
            -100,
            device="cuda",
            dtype=torch.long,
        )

        original_target_tokens = train_part[:, -encode_size:]

        new_target_labels = torch.cat(
            [
                original_target_tokens[:, 0].unsqueeze(1),
                original_target_tokens,
            ],
            dim=1,
        )

        labels_part[:, context_len : context_len + target_len] = new_target_labels

        compression_tensor_param = torch.nn.Parameter(
            torch.rand(
                mem_tokens,
                model.get_input_embeddings().weight.shape[1],
                device="cuda",
            ).unsqueeze(0),
            requires_grad=True,
        )
        optimizer = torch.optim.AdamW(
            [compression_tensor_param],
            lr=0.1,
        )

        prev_tokens = train_part[:, :-encode_size].clone()
        prev_embeds = model.get_input_embeddings()(prev_tokens)

        epoch_amount = 50
        dtype = torch.bfloat16
        for epoch in tqdm(range(epoch_amount)):
            compression_tensor = compression_tensor_param.repeat(1, num_repeats, 1)

            input_embeds = torch.cat(
                [
                    prev_embeds,
                    compression_tensor,
                ],
                dim=1,
            ).to(dtype)

            assert input_embeds.shape[1] == labels_part.shape[1]

            model_predicts = model(
                inputs_embeds=input_embeds,
                labels=labels_part,
            )

            compression_loss = model_predicts.loss
            compression_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        # --- НАЧАЛО ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---

        # 1. Эталонная последовательность для проверки - это все метки, КРОМЕ ПОСЛЕДНЕЙ.
        #    Ее длина `encode_size`.
        check_target_labels = new_target_labels[0, 1:]
        check_target_str = tokenizer.decode(
            check_target_labels, add_special_tokens=False
        )

        # 2. Извлекаем предсказания модели. Нам нужна та же длина, что и у эталона.
        predicted_tokens = model_predicts.logits.argmax(-1)
        # Берем срез длиной `target_len`, как и раньше...
        predicted_target_part = predicted_tokens[
            :, context_len : context_len + target_len
        ][0]

        # 3. ...но для сравнения отбрасываем последний токен.
        prediction_str = tokenizer.decode(
            predicted_target_part[:-1], add_special_tokens=False
        )

        # 4. Сравниваем строки длиной `encode_size`.
        correct_reconstruction = prediction_str == check_target_str

        # --- КОНЕЦ ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---
        print(f"FULL TEXT: '{tokenizer.decode(train_part[:, :][-1])}'")
        print("-" * 50)
        print(
            f"Context: '{tokenizer.decode(train_part[:, :-encode_size][-1])}'",
        )
        # Эталонная строка, которую модель должна была сгенерировать
        print(
            f"TARGET to generate: '{check_target_str}'",
        )
        # Строка, которую модель сгенерировала на самом деле
        print(f"PREDICTED string:   '{prediction_str}'")
        print(f"Correct reconstruction: '{correct_reconstruction}'")
        print("=" * 50)
        print("=" * 50)
        cramming_tokens.append(compression_tensor_param.detach())
        # break

    break

100/209


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 35.30it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
TARGET to generate: ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
PREDICTED string:   ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
Correct reconstruction: 'True'


100%|██████████| 50/50 [00:01<00:00, 35.98it/s]


...

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me thin

100%|██████████| 50/50 [00:01<00:00, 33.30it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the'
--------------------------------------

100%|██████████| 50/50 [00:01<00:00, 32.99it/s]

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the problem is trying to trick me into thin

### Decode part

In [41]:
import torch
from more_itertools import chunked
import itertools

# --- ПРЕДПОЛАГАЕТСЯ, ЧТО ЭТИ ОБЪЕКТЫ УЖЕ СУЩЕСТВУЮТ В ВАШЕЙ СРЕДЕ ---
# model: ваша языковая модель, уже загруженная на GPU
# tokenizer: ваш токенизатор
# dataset: ваш датасет
# cramming_tokens: список с обученными тензорами
# --------------------------------------------------------------------------


def predict_in_one_pass(
    model,
    tokenizer,
    context_tokens: torch.Tensor,
    learned_compression_tensor: torch.Tensor,
    expected_len: int,
):
    """
    Выполняет один прямой проход и извлекает предсказанные токены
    с позиций, соответствующих сжимающему тензору.

    Args:
        model: Языковая модель.
        tokenizer: Токенизатор.
        context_tokens (torch.Tensor): Токены контекста, shape [1, context_len].
        learned_compression_tensor (torch.Tensor): Обученный тензор, shape [1, mem_tokens, hidden_size].
        expected_len (int): Длина ожидаемой последовательности (encode_size).

    Returns:
        str: Распакованный текст.
    """
    print("--- Запуск инференса в один проход ---")

    model.eval()
    device = model.device
    dtype = model.dtype

    context_tokens = context_tokens.to(device)
    # Важно: здесь мы повторяем сжимающий тензор точно так же, как в обучении!
    # Это необходимо, чтобы на выходе было достаточно позиций для предсказания
    # всей последовательности длиной `encode_size`.
    mem_tokens = learned_compression_tensor.shape[1]
    target_len = expected_len + 1  # +1, как в вашем коде обучения для new_target_labels
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    repeated_compression_tensor = learned_compression_tensor.repeat(
        1, num_repeats, 1
    ).to(dtype)

    with torch.no_grad():
        # 1. Формируем входные эмбеддинги: контекст + повторенный сжимающий тензор
        context_embeds = model.get_input_embeddings()(context_tokens)
        input_embeds = torch.cat(
            [context_embeds, repeated_compression_tensor],
            dim=1,
        )

        # 2. Делаем ОДИН прямой проход через модель
        outputs = model(inputs_embeds=input_embeds)

        # 3. Извлекаем логиты. Нас интересуют предсказания, которые модель делает,
        #    "прочитав" контекст и начав "читать" сжимающий тензор.
        #    В стандартных causal-моделях transformers, logit[i] предсказывает token[i+1].
        #    Поэтому, чтобы предсказать первую часть целевой последовательности, мы берем
        #    логиты начиная с конца контекста.
        context_len = context_embeds.shape[1]

        # Срезаем логиты с позиций, соответствующих нашему сжатому тензору.
        # Длина среза - ровно `expected_len` (encode_size).
        predicted_logits = outputs.logits[
            :, context_len : context_len + expected_len, :
        ]

        # 4. Находим наиболее вероятные токены для каждой позиции
        predicted_token_ids = torch.argmax(
            predicted_logits, dim=-1
        )  # Shape: [1, expected_len]

        # 5. Декодируем ID токенов в текст
        reconstructed_text = tokenizer.decode(
            predicted_token_ids[0], skip_special_tokens=True
        )

    return reconstructed_text


# --- ПРИМЕР ИСПОЛЬЗОВАНИЯ (остается таким же) ---
# Проверяем, что в `cramming_tokens` есть хотя бы один обученный тензор
if not cramming_tokens:
    raise ValueError(
        "Список `cramming_tokens` пуст. Сначала запустите скрипт обучения."
    )

# Параметры, идентичные обучающему скрипту
start_item = 100
mem_tokens = 8
encode_size = mem_tokens * 4
chunk_part_to_test = 2

# 1. Получаем те же самые данные, что были при обучении
item_index = start_item
tokens = tokenizer.encode(
    dataset[item_index]["model_answer"],
    add_special_tokens=False,
)
tokens_chunks = list(chunked(tokens, encode_size))

# 2. Формируем контекст и эталонный (целевой) текст
train_part_chunks = tokens_chunks[:chunk_part_to_test]
train_part_list = list(itertools.chain(*train_part_chunks))

original_target_tokens_list = train_part_list[-encode_size:]
context_tokens_list = train_part_list[:-encode_size]

# 3. Преобразуем списки токенов в тензоры
context_tokens_tensor = torch.tensor(
    [context_tokens_list],
    dtype=torch.long,
)
original_target_text = tokenizer.decode(original_target_tokens_list)

# 4. Запускаем инференс с помощью новой функции
learned_tensor_to_test = cramming_tokens[0]

reconstructed_text = predict_in_one_pass(
    model=model,
    tokenizer=tokenizer,
    context_tokens=context_tokens_tensor,
    learned_compression_tensor=learned_tensor_to_test,
    expected_len=encode_size,
)

# 5. Выводим результаты для сравнения
print("\n" + "=" * 80)
print(
    f"Контекст, который был подан в модель: '{tokenizer.decode(context_tokens_list)}'"
)
print("-" * 80)
print(f"ЭТАЛОН (что должны были восстановить):")
print(f"'{original_target_text}'")
print("-" * 80)
print(f"РЕЗУЛЬТАТ (что сгенерировала модель):")
print(f"'{reconstructed_text}'")
print("=" * 80)

if original_target_text.strip() == reconstructed_text.strip():
    print("\n✅ Успех! Текст восстановлен корректно.")
else:
    print("\n❌ Неудача. Текст восстановлен с ошибками.")

--- Запуск инференса в один проход ---

Контекст, который был подан в модель: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
--------------------------------------------------------------------------------
ЭТАЛОН (что должны были восстановить):
' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
--------------------------------------------------------------------------------
РЕЗУЛЬТАТ (что сгенерировала модель):
' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'

✅ Успех! Текст восстановлен корректно.
